# Per-category Diagnostic — Nemotron LoRA adapter

Loads a pretrained adapter and runs maj@N inference on a stratified subset of `train.csv` to find weak categories. Output: `/kaggle/working/diagnostic_report.csv`.

In [ ]:
# ============================================================
# DIAGNOSTIC CONFIG — fill these in before running on Kaggle
# ============================================================
# Pretrained LoRA adapter (must be a Kaggle dataset you attach to this notebook).
# Path layout: /kaggle/input/datasets/{owner}/{slug}/ containing adapter_config.json + adapter_model.safetensors
PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/dgxchen/trained-adapter"  # TODO: replace with your best adapter

# Diagnostic subset CSV (3-column: prompt, answer, category) attached as a Kaggle dataset.
DIAGNOSTIC_SUBSET_PATH = "/kaggle/input/datasets/hiranorm/nemotron-diagnostic-subset/diagnostic_subset.csv"

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

# Inference cost knobs. Default = greedy (maj@1) to keep the run tractable on Kaggle GPU budgets.
# Bump MAJ_N to 8 (or higher) if you want a closer proxy of the maj@64 eval, but expect proportional time.
MAJ_N = 8              # 1 = greedy; >1 = sample MAJ_N completions and majority-vote the boxed answer
MAX_PROBLEMS_PER_CAT = 30  # subset-of-subset cap (the input subset has ~60/category)
MAX_NEW_TOKENS = 1536
TEMPERATURE = 0.7      # only used when MAJ_N > 1
TOP_P = 0.9

import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")

import random, numpy as np, torch
GLOBAL_SEED = 777
random.seed(GLOBAL_SEED); np.random.seed(GLOBAL_SEED); torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)

# We always run in inference mode; reuse upstream flag names so the shared setup cells branch correctly.
TRAIN_ON_KAGGLE = 1   # keep the wheel/Unsloth setup cells active
USE_PRETRAINED = 0    # we load the adapter manually in the diagnostic cells below

print({"adapter": PRETRAINED_ADAPTER_DATASET_PATH, "subset": DIAGNOSTIC_SUBSET_PATH,
       "MAJ_N": MAJ_N, "MAX_PROBLEMS_PER_CAT": MAX_PROBLEMS_PER_CAT})


## Environment setup (reused from run-EXP002-child-exp001.ipynb)

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)

site.addsitedir(target)

print("Custom target added:", target)

import importlib.util
print("triton spec：", importlib.util.find_spec("triton"))


In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    # Add utility script to Python path (provides helper binaries)
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    # Copy ptxas-blackwell to /tmp with execute permissions
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")


In [ ]:
# trl installation is handled by the Unsloth offline setup cell below.
if TRAIN_ON_KAGGLE:
    print("Skip standalone trl install/import here; the Unsloth setup cell will install compatible packages.")

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob
    import os
    import subprocess
    import sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA:", torch.version.cuda)

    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime because Nemotron depends on CUDA wheels.")

    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", packages_dir,
            "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes",
        ],
        check=True,
    )

    def pick_last(wheels):
        return wheels[-1] if wheels else None

    causal_wheel = pick_last(all_causal)
    mamba_wheel = pick_last(all_mamba)
    print("Selected causal wheel:", causal_wheel)
    print("Selected mamba wheel:", mamba_wheel)

    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("Could not find a compatible mamba_ssm wheel under /kaggle/input.")

    print("Offline package installation finished. Restart the kernel if Kaggle keeps stale imports from earlier runs.")
else:
    print("USE_PRETRAINED=1: skipping datasets / trl / mamba_ssm / unsloth installation.")


In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="sdpa",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")


## Load pretrained adapter

In [ ]:
# Load the pretrained LoRA adapter onto the base model (replaces the SFTTrainer training step).
import os, glob, json
from peft import PeftModel

ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
if not os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
    # fallback: search anywhere under /kaggle/input
    candidates = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if not candidates:
        raise FileNotFoundError(f"adapter_config.json not found at {ADAPTER_DIR} nor under /kaggle/input")
    ADAPTER_DIR = os.path.dirname(candidates[0])
    print("Auto-discovered adapter dir:", ADAPTER_DIR)

print("Loading adapter from:", ADAPTER_DIR)
for fname in ["adapter_config.json", "adapter_model.safetensors"]:
    fp = os.path.join(ADAPTER_DIR, fname)
    print(f"  {fname}: {os.path.getsize(fp)/1024/1024:.1f} MB")

# `model` from the upstream setup cell is the bare base. Wrap it with the adapter.
model = PeftModel.from_pretrained(model, ADAPTER_DIR, is_trainable=False)
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)
model.eval()
print("Adapter loaded and model switched to inference mode.")


## Inference and per-category accuracy

In [ ]:
# Per-category diagnostic inference.
import os, re, time, json, glob, pathlib
from collections import Counter, defaultdict

import pandas as pd
import torch

PROMPT_SUFFIX = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
BOXED_RE = re.compile(r"\\boxed\{([^{}]*)\}")

subset_path = DIAGNOSTIC_SUBSET_PATH
if not os.path.exists(subset_path):
    cands = sorted(glob.glob("/kaggle/input/**/diagnostic_subset.csv", recursive=True))
    if not cands:
        raise FileNotFoundError(f"diagnostic_subset.csv not found at {subset_path} nor under /kaggle/input")
    subset_path = cands[0]
    print("Auto-discovered subset:", subset_path)

df = pd.read_csv(subset_path)
assert {"prompt", "answer", "category"}.issubset(df.columns), df.columns
print("Loaded subset:", len(df), "rows. Per-category:")
print(df["category"].value_counts().to_dict())

# Cap per category.
df = df.groupby("category", group_keys=False).apply(
    lambda g: g.head(MAX_PROBLEMS_PER_CAT)
).reset_index(drop=True)
print("After cap:", len(df), "rows")


def extract_boxed(text: str) -> str:
    matches = BOXED_RE.findall(text or "")
    return matches[-1].strip() if matches else ""


def render_prompt(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt + PROMPT_SUFFIX}]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def generate_one(text: str) -> str:
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    do_sample = MAJ_N > 1
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=do_sample,
            temperature=TEMPERATURE if do_sample else 1.0,
            top_p=TOP_P if do_sample else 1.0,
            pad_token_id=tokenizer.pad_token_id,
        )
    new = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True)


def majority_predict(prompt: str) -> tuple[str, list[str]]:
    text = render_prompt(prompt)
    votes = []
    for _ in range(MAJ_N):
        gen = generate_one(text)
        votes.append(extract_boxed(gen))
    nonempty = [v for v in votes if v]
    if not nonempty:
        return "", votes
    pred, _ = Counter(nonempty).most_common(1)[0]
    return pred, votes


t0 = time.time()
per_cat_correct = defaultdict(int)
per_cat_total = defaultdict(int)
rows = []
for i, r in df.iterrows():
    cat = r["category"]
    gold = str(r["answer"]).strip()
    pred, votes = majority_predict(str(r["prompt"]))
    ok = (pred == gold)
    per_cat_total[cat] += 1
    per_cat_correct[cat] += int(ok)
    rows.append({"category": cat, "gold": gold, "pred": pred, "ok": int(ok), "votes": json.dumps(votes)})
    if (i + 1) % 10 == 0:
        elapsed = time.time() - t0
        print(f"[{i+1}/{len(df)}] cat={cat} ok={ok} elapsed={elapsed:.1f}s")

print("\n=== Per-category accuracy ===")
report_rows = []
for cat in sorted(per_cat_total):
    tot = per_cat_total[cat]
    cor = per_cat_correct[cat]
    acc = cor / tot if tot else 0.0
    print(f"  {cat:10s}: {cor:3d}/{tot:3d} = {acc:.3f}")
    report_rows.append({"category": cat, "correct": cor, "total": tot, "accuracy": acc})

OUT_DIR = "/kaggle/working"
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "diagnostic_per_sample.csv"), index=False)
pd.DataFrame(report_rows).to_csv(os.path.join(OUT_DIR, "diagnostic_report.csv"), index=False)
print("\nSaved:")
print("  /kaggle/working/diagnostic_per_sample.csv")
print("  /kaggle/working/diagnostic_report.csv")
print(f"Total elapsed: {time.time()-t0:.1f}s, MAJ_N={MAJ_N}, samples={len(df)}")
